# 07 Dostrajanie modelu NLI (H4a)

## 1. Konfiguracja

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/Licencjat/extra

import sys
sys.path.append("..")

import gc

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score, f1_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from src.config import (FINETUNE_BATCH_SIZE, FINETUNE_EPOCHS, FINETUNE_FOLDS, FINETUNE_LEARNING_RATE,
                        FINETUNE_WARMUP_RATIO, FINETUNED_MODEL_DIR, LABELS_DIR, NLI_HYPOTHESIS, NLI_MODEL,
                        PROCESSED_DIR, RANDOM_STATE, RESULTS_DIR)
from src.nli import nli_probabilities

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Brak GPU: Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU"
torch.cuda.get_device_name(0)

## 2. Etykiety i teksty (`labels/stance_train.csv`, `labels/stance_test.csv`: `id`, `term`, `label`)

In [ ]:
mentions = pd.read_parquet(PROCESSED_DIR / "h4a" / "mentions.parquet", columns=["id", "term", "text", "target", "cycle"])
labelled = {}
for name in ("train", "test"):
    labels = pd.read_csv(LABELS_DIR / f"stance_{name}.csv", dtype={"id": str})
    labelled[name] = labels.merge(mentions, on=["id", "term"])
    print(name, "etykiet:", len(labels), "| znalezionych tekstów:", len(labelled[name]))
train, test = labelled["train"], labelled["test"]
pd.crosstab(train["target"], train["label"])

## 3. Trening i ocena

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)


def train_model(texts, terms, labels):
    torch.manual_seed(RANDOM_STATE)
    model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)
    label_ids = {name.lower(): index for name, index in model.config.label2id.items()}
    # etykieta 1 (krytyka) uczy klasy entailment, etykieta 0 – klasy contradiction
    class_of = {1: label_ids["entailment"], 0: label_ids["contradiction"]}
    hypotheses = [NLI_HYPOTHESIS.format(term) for term in terms]
    steps = FINETUNE_EPOCHS * int(np.ceil(len(texts) / FINETUNE_BATCH_SIZE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=FINETUNE_LEARNING_RATE)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(FINETUNE_WARMUP_RATIO * steps), steps)
    scaler = torch.amp.GradScaler()
    rng = np.random.default_rng(RANDOM_STATE)
    model.train()
    for epoch in range(FINETUNE_EPOCHS):
        order = rng.permutation(len(texts))
        for start in tqdm(range(0, len(order), FINETUNE_BATCH_SIZE), desc=f"epoka {epoch + 1}"):
            index = order[start:start + FINETUNE_BATCH_SIZE]
            batch = tokenizer([texts[i] for i in index], [hypotheses[i] for i in index], truncation="only_first",
                              max_length=512, padding=True, return_tensors="pt").to(device)
            targets = torch.tensor([class_of[labels[i]] for i in index], device=device)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**batch).logits
            loss = torch.nn.functional.cross_entropy(logits.float(), targets)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
    return model.eval()


def score(data):
    sensitivity = data.loc[data["label"] == 1, "predicted"].mean()
    false_alarm = data.loc[data["label"] == 0, "predicted"].mean()
    return {"n": len(data), "kappa": cohen_kappa_score(data["label"], data["predicted"]),
            "f1": f1_score(data["label"], data["predicted"]), "sensitivity": sensitivity,
            "false_alarm": false_alarm, "sensitivity_minus_false_alarm": sensitivity - false_alarm}


def score_table(data, by):
    rows = [{"group": "all", **score(data)}]
    for column in by:
        for value, group in data.groupby(column):
            rows.append({"group": f"{column}={value}", **score(group)})
    return pd.DataFrame(rows)

## 4. Walidacja krzyżowa na zbiorze treningowym

In [ ]:
strata = train["label"].astype(str) + "_" + train["target"]
folds = StratifiedKFold(n_splits=FINETUNE_FOLDS, shuffle=True, random_state=RANDOM_STATE)
train["predicted"] = -1
fold_scores = []
for fold, (fit_index, held_index) in enumerate(folds.split(train, strata), start=1):
    fit, held = train.loc[fit_index], train.loc[held_index]
    model = train_model(fit["text"].tolist(), fit["term"].tolist(), fit["label"].tolist())
    probabilities = nli_probabilities(held["text"].tolist(), held["term"].tolist(), model, tokenizer, device, FINETUNE_BATCH_SIZE)
    train.loc[held_index, "predicted"] = probabilities["critical"].astype(int).values
    fold_scores.append({"fold": fold, **score(train.loc[held_index])})
    del model
    gc.collect()
    torch.cuda.empty_cache()
fold_scores = pd.DataFrame(fold_scores)
fold_scores.to_csv(RESULTS_DIR / "validation_stance_cv_folds.csv", index=False)
fold_scores.round(3)

In [ ]:
cv_scores = score_table(train, ["target"])
cv_scores.to_csv(RESULTS_DIR / "validation_stance_cv.csv", index=False)
cv_scores.round(3)

## 5. Trening końcowy na całym zbiorze treningowym

In [ ]:
model = train_model(train["text"].tolist(), train["term"].tolist(), train["label"].tolist())
FINETUNED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(FINETUNED_MODEL_DIR)
tokenizer.save_pretrained(FINETUNED_MODEL_DIR)

## 6. Ocena na zbiorze testowym

In [ ]:
probabilities = nli_probabilities(test["text"].tolist(), test["term"].tolist(), model, tokenizer, device, FINETUNE_BATCH_SIZE)
test["predicted"] = probabilities["critical"].astype(int).values
test_scores = score_table(test, ["target", "cycle"])
test_scores.to_csv(RESULTS_DIR / "validation_stance_test.csv", index=False)
test_scores.round(3)